# 7. PPI (Product Problem Information) Ingestion

This notebook runs the PPI dual-write ingestion pipeline:
1. **Migration**: Add PPI columns to `emr_records` table (PostgreSQL) — idempotent
2. **Ingestion**: Read Excel → PostgreSQL UPDATE + Neo4j MERGE → relationships
3. **Verification**: Count checks, sample data, sync consistency

In [ ]:
import os, sys
cwd = os.path.abspath(os.getcwd())
project_root = None
temp = cwd
for _ in range(4):
    if os.path.exists(os.path.join(temp, 'src', 'config.py')):
        project_root = temp
        break
    parent = os.path.abspath(os.path.join(temp, '..'))
    if parent == temp:
        break
    temp = parent
if project_root and project_root not in sys.path:
    sys.path.insert(0, project_root)
    print(f"Project root: {project_root}")
else:
    print("Warning: using default path")
    sys.path.insert(0, os.path.join(cwd, '..'))

## Step 1: Migration — Add PPI Columns to PostgreSQL

Idempotent: `ALTER TABLE ... ADD COLUMN IF NOT EXISTS` — safe to re-run.

In [ ]:
from scripts.migrate_ppi import run_migration, MIGRATE_SQL, MIGRATE_INDEXES
print("Migration SQL:")
for sql in MIGRATE_SQL:
    print(f"  {sql}")
print("\nIndexes:")
for idx in MIGRATE_INDEXES:
    print(f"  {idx}")
print("\nRunning migration...")
run_migration()
print("\nMigration complete!")

## Step 2: Dry Run — Preview Without Writes

Check what data would be ingested.

In [ ]:
from scripts.ingest_ppi import ingest_ppi
result = ingest_ppi(dry_run=True)
print(f"\nDry run complete. Would process {result['total_rows']} EMRs.")

## Step 3: Full Ingestion — Dual Write

PostgreSQL UPDATE + Neo4j MERGE PPI nodes + relationships.

In [ ]:
result = ingest_ppi(dry_run=False)
print(f"""
Ingestion complete:
  PG updated:     {result['pg_updated']}
  Neo4j merged:   {result['neo4j_merged']} nodes + {result['neo4j_relations']} relations
  Skipped:        {result['skipped_emr_not_found']}
  Duration:       {result['duration_sec']}s
""")

## Step 4: Verification

Verify data integrity across both databases.

In [ ]:
from sqlalchemy import create_engine, text
from src.config import settings

pg_engine = create_engine(settings.postgres_url)
with pg_engine.connect() as conn:
    row = conn.execute(text("""
        SELECT COUNT(*) AS total,
               COUNT(ppi_external_id) AS has_ppi,
               COUNT(DISTINCT ppi_external_id) AS unique_ppi
        FROM emr_records
    """)).fetchone()
    print(f"PostgreSQL emr_records:")
    print(f"  Total rows:  {row.total}")
    print(f"  With PPI:    {row.has_ppi}")
    print(f"  Unique PPIs: {row.unique_ppi}")

    sample = conn.execute(text("""
        SELECT emr_name, ppi_external_id, ppi_improvement_name
        FROM emr_records
        WHERE ppi_external_id IS NOT NULL
        LIMIT 5
    """)).fetchall()
    print("\nSample records with PPI:")
    for r in sample:
        print(f"  {r.emr_name} → {r.ppi_external_id}: {r.ppi_improvement_name}")

In [ ]:
from src.graph.client import GraphClient
gc = GraphClient(settings.neo4j_uri, settings.neo4j_user, settings.neo4j_password)
try:
    # Count PPI nodes
    node_count = gc.run_query("MATCH (p:PPI) RETURN count(p) AS count")
    print(f"Neo4j PPI nodes: {node_count[0]['count']}")

    # Count HAS_PPI relationships
    rel_count = gc.run_query("MATCH (:EMRRecord)-[r:HAS_PPI]->(:PPI) RETURN count(r) AS count")
    print(f"Neo4j HAS_PPI relationships: {rel_count[0]['count']}")

    # Sample PPI nodes
    sample = gc.run_query("""
        MATCH (p:PPI)
        RETURN p.external_id, p.improvement_name, p.phenomenon
        LIMIT 5
    """)
    print("\nSample PPI nodes:")
    for r in sample:
        print(f"  {r['p.external_id']}: {r['p.improvement_name']} — {r['p.phenomenon']}")
finally:
    gc.close()